In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

In [3]:
train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(train['Age'].median())
train['Fare'] = train['Fare'].fillna(train['Fare'].median())
test['Fare'] = test['Fare'].fillna(train['Fare'].median())
train['Embarked'] = train['Embarked'].fillna('S')
test['Embarked'] = test['Embarked'].fillna('S')

In [4]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

In [5]:
train = pd.get_dummies(train, columns=['Sex','Embarked'])
test = pd.get_dummies(test, columns=['Sex','Embarked'])

In [6]:
train, test = train.align(test, join='left', axis=1, fill_value=0)
test = test.reindex(columns=train.columns, fill_value=0)

In [7]:
X = train[['Pclass','Age','Fare', 'FamilySize', 'IsAlone', 'Sex_male','Embarked_Q','Embarked_S']]
y = train['Survived']
X_test = test[['Pclass','Age','Fare','FamilySize', 'IsAlone', 'Sex_male','Embarked_Q','Embarked_S']]

In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42)
model.fit(X, y)

RandomForestClassifier(max_depth=5, min_samples_leaf=5, min_samples_split=10,
                       n_estimators=300, random_state=42)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)
train_preds = model.predict(X_train)
train_acc = accuracy_score(y_train, train_preds)
val_pred = model.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_pred))
print("Train Accuracy:", train_acc)

Validation accuracy: 0.8100558659217877
Train Accuracy: 0.8567415730337079


In [10]:
predictions = model.predict(X_test)
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)